# Aufbau Wissensgraph mit RDF-Lib

In [ ]:
import pandas as pd
import kagglehub
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")

print("Path to dataset files:", path)
data_path = path+"\World-Stock-Prices-Dataset.csv"
data = pd.read_csv(data_path)

<>:11: SyntaxWarning: invalid escape sequence '\W'
<>:11: SyntaxWarning: invalid escape sequence '\W'
C:\Users\s3phi\AppData\Local\Temp\ipykernel_16436\2468708934.py:11: SyntaxWarning: invalid escape sequence '\W'
  data_path = path+"\World-Stock-Prices-Dataset.csv"
c:\workspace\StockPredictor\StockPredictor_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\s3phi\.cache\kagglehub\datasets\nelgiriyewithana\world-stock-prices-daily-updating\versions\389


In [ ]:
brands = []
arr = data["Brand_Name"].unique()
for i in arr:
    brands.append(i.replace(" ", "-"))

brands

In [ ]:
df_brands = pd.DataFrame({"brands" : brands})
wikidata_elements = ["Q56276186", "Q926699", "Q3295867", "Q3895", "Q194360", "Q157064", "Q328840", "Q11463", "Q157062", "Q173395", "Q192314", "Q504998", "Q63327", "Q1141173", "Q8074134", "Q53268", "Q1057464", 
                     "Q38076", "Q864407", "Q489921", "Q333718", "Q780442", "Q212405", "Q16972754", "Q459477", "Q159433", "Q170416", "Q63335", "Q907311", "Q128896", "Q188273", "Q7501150", "Q503308", "Q223127", 
                     "Q3884", "Q312", "Q483915", "Q1046951", "Q95", "Q689141", "Q17460900", "Q7414", "Q67186598", "Q8093", "Q188920", "Q2283", "Q715583", "Q2842931", "Q609466", "Q96095585", "Q26678", 
                     "Q465751", "Q868666", "Q40993", "Q9584", "Q941127", "Q182477", "Q37158", "Q478214", "Q918", "Q174310", "Q30258651"]
df_brands["wikidata"] = wikidata_elements
df_brands.tail(20)

,brand,wikidata
42,roblox,Q67186598
43,nintendo,Q8093
44,delta-air-lines,Q188920
45,microsoft,Q2283
46,costco,Q715583
47,american-eagle-outfitters,Q2842931
48,colgate-palmolive,Q609466
49,pinterest,Q96095585
50,bmw-group,Q26678
51,chipotle,Q465751


In [ ]:
company = df_brands.loc[df_brands["brands"] == "tesla"]
brand_identifier = company["wikidata"].loc[company.index[0]]
print(brand_identifier)


Q478214


In [170]:
query_start = "SELECT ?officialname ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization\n" \
"WHERE {"

query_order = "wd:"+brand_identifier+" wdt:P1448 ?officialname;\n wdt:P154 ?logo;\n wdt:P571 ?inception;\n wdt:P2403 ?totalassets;\n wdt:P2139 ?revenue;\n wdt:P2295 ?netprofit;\n wdt:P3362 ?operatingincome;\n wdt:P2226 ?marketcapitalization.\n"

query_end = "}"

query1 = query_start+query_order+query_end

print(query1)

SELECT ?officialname ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization
WHERE {wd:Q478214 wdt:P1448 ?officialname;
 wdt:P154 ?logo;
 wdt:P571 ?inception;
 wdt:P2403 ?totalassets;
 wdt:P2139 ?revenue;
 wdt:P2295 ?netprofit;
 wdt:P3362 ?operatingincome;
 wdt:P2226 ?marketcapitalization.
}


In [171]:
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
# From https://www.wikidata.org/wiki/Wikidata:SPARQL_query_service/queries/examples#Cats
sparql.setQuery(query1)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

In [172]:
results

{'head': {'vars': ['officialname',
   'logo',
   'inception',
   'totalassets',
   'revenue',
   'netprofit',
   'operatingincome',
   'marketcapitalization']},
 'results': {'bindings': [{'officialname': {'xml:lang': 'en',
     'type': 'literal',
     'value': 'Tesla, Inc.'},
    'logo': {'type': 'uri',
     'value': 'http://commons.wikimedia.org/wiki/Special:FilePath/Tesla%20T%20symbol.svg'},
    'inception': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
     'type': 'literal',
     'value': '2003-07-01T00:00:00Z'},
    'totalassets': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
     'type': 'literal',
     'value': '62131000000'},
    'revenue': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
     'type': 'literal',
     'value': '97690000000'},
    'netprofit': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
     'type': 'literal',
     'value': '7091000000'},
    'operatingincome': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
  

In [78]:
print(results['results']['bindings'][0]["officialname"]["value"])
print(results['results']['bindings'][0]["inception"]["value"])
print(results['results']['bindings'][0]["totalassets"]["value"],"€")
print(results['results']['bindings'][0]["revenue"]["value"],"€")
print(results['results']['bindings'][0]["netprofit"]["value"],"€")
print(results['results']['bindings'][0]["operatingincome"]["value"],"€")
print(results['results']['bindings'][0]["marketcapitalization"]["value"],"€")
n = int(results['results']['bindings'][0]["marketcapitalization"]["value"])
res = "{:,}".format(n)
print(res,"€")

Bayerische Motoren Werke AG
1916-03-07T00:00:00Z
228034000000 €
142610000000 €
7680000000 €
13999000000 €
59907000000 €
59,907,000,000 €


In [79]:
results_df = pd.DataFrame(results['results']['bindings'])
results_df

,officialname,logo,inception,totalassets,revenue,netprofit,operatingincome,marketcapitalization
0,"{'xml:lang': 'de', 'type': 'literal', 'value':...","{'type': 'uri', 'value': 'http://commons.wikim...",{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...


<div class="alert alert-block alert-info">
<b>Zusammenfassung für Gradio</b> 
</div>

In [175]:
import pandas as pd
import kagglehub
from SPARQLWrapper import SPARQLWrapper, JSON

path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")

print("Path to dataset files:", path)
data_path = path+"\World-Stock-Prices-Dataset.csv"
data = pd.read_csv(data_path)

<>:8: SyntaxWarning: invalid escape sequence '\W'
<>:8: SyntaxWarning: invalid escape sequence '\W'
C:\Users\s3phi\AppData\Local\Temp\ipykernel_16436\1466582878.py:8: SyntaxWarning: invalid escape sequence '\W'
  data_path = path+"\World-Stock-Prices-Dataset.csv"


Path to dataset files: C:\Users\s3phi\.cache\kagglehub\datasets\nelgiriyewithana\world-stock-prices-daily-updating\versions\391


In [196]:
brands = []
arr = data["Brand_Name"].unique() #gebe jede firma ohne dubletten an
for i in arr:
    brands.append(i.replace(" ", "-")) #lösche leerzeichen und ersetze diese durch bindestriche

df_brands = pd.DataFrame({"brands" : brands}) #neues data frame erstellen mir den firmen
wikidata_elements = ["Q56276186", "Q926699", "Q3295867", "Q3895", "Q194360", "Q157064", "Q328840", "Q11463", "Q157062", "Q173395", "Q192314", "Q504998", "Q63327", "Q1141173", "Q8074134", "Q53268", "Q1057464", 
                     "Q38076", "Q864407", "Q489921", "Q333718", "Q780442", "Q212405", "Q16972754", "Q459477", "Q159433", "Q170416", "Q63335", "Q907311", "Q128896", "Q188273", "Q7501150", "Q503308", "Q223127", 
                     "Q3884", "Q312", "Q483915", "Q1046951", "Q95", "Q689141", "Q17460900", "Q7414", "Q67186598", "Q8093", "Q188920", "Q2283", "Q715583", "Q2842931", "Q609466", "Q96095585", "Q26678", 
                     "Q465751", "Q868666", "Q40993", "Q9584", "Q941127", "Q182477", "Q37158", "Q478214", "Q918", "Q174310", "Q30258651"]
df_brands["wikidata"] = wikidata_elements #füge die wikidata identifier als neue spalte hinzu

company = df_brands.loc[df_brands["brands"] == "bmw-group"]
brand_identifier = company["wikidata"].loc[company.index[0]]

query_start = "SELECT ?officialname ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization\n WHERE {"
query_order = "wd:"+brand_identifier+" wdt:P1448 ?officialname;\n wdt:P154 ?logo;\n wdt:P571 ?inception;\n wdt:P2403 ?totalassets;\n wdt:P2139 ?revenue;\n wdt:P2295 ?netprofit;\n wdt:P3362 ?operatingincome;\n wdt:P2226 ?marketcapitalization.\n"
query_end = "}"

query1 = query_start+query_order+query_end

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
# From https://www.wikidata.org/wiki/Wikidata:SPARQL_query_service/queries/examples#Cats
sparql.setQuery(query1)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print(results['results']['bindings'][0]["officialname"]["value"])
print(results['results']['bindings'][0]["logo"]["value"])
print(results['results']['bindings'][0]["inception"]["value"])
print(results['results']['bindings'][0]["totalassets"]["value"],"€")
print(results['results']['bindings'][0]["revenue"]["value"],"€")
print(results['results']['bindings'][0]["netprofit"]["value"],"€")
print(results['results']['bindings'][0]["operatingincome"]["value"],"€")
print(results['results']['bindings'][0]["marketcapitalization"]["value"],"€")
marketcaptl = "{:,}".format(int(results['results']['bindings'][0]["marketcapitalization"]["value"]))+" €"
print(marketcaptl)

Bayerische Motoren Werke AG
http://commons.wikimedia.org/wiki/Special:FilePath/BMW%20logo%20%28gray%29.svg
1916-03-07T00:00:00Z
228034000000 €
142610000000 €
7680000000 €
13999000000 €
59907000000 €
59,907,000,000 €


In [195]:
inception = results['results']['bindings'][0]["inception"]["value"]
inception_short = inception[:10]
inception_short

'2003-07-01'